PyTorch 的自动求导机制（Autograd）是深度学习的“魔法引擎”。
- 简单来说，<font color='red'>它是一个记录张量操作历史的系统</font>，能够自动帮你计算复杂的导数（梯度），从而免去手写反向传播的痛苦。

要彻底理解它，你只需要掌握三个核心概念：
- **张量的属性**
- **计算图（DAG）**
- **反向传播。**

#  核心开关：requires_grad 与 grad_fn

Autograd 并不是对所有张量都求导，<font color='red'>它只追踪那些被标记为“需要求导”的张量。</font>
- requires_grad=True：这是一个开关。
  - **一旦某个张量被设为 True，PyTorch 就会开始记录所有在这个张量上发生的操作**。
- grad_fn：这是操作的“<font color='red'>记账员</font>”。如果一个张量是由其他张量经过运算得来的，它的 grad_fn 属性就会指向生成它的那个操作（例如 <AddBackward> <MulBackward>）。
    
**注：**用户直接创建的张量（叶子节点），其 grad_fn 为 None。

In [1]:
import torch

# 1. 创建叶子节点，并开启求导
x = torch.tensor([2.0], requires_grad=True)
w = torch.tensor([3.0], requires_grad=True)

# 2. 进行运算
y = x * w       # y 是由 x 和 w 相乘得到的
z = y + 5       # z 是由 y 加上常数得到的

print(y.grad_fn)  # 输出: <MulBackward0 object>
print(z.grad_fn)  # 输出: <AddBackward0 object>
print(x.grad_fn)  # 输出: None (因为 x 是叶子节点)

None


# 底层逻辑：动态计算图 (DAG)

Autograd 在底层构建的是一个有向无环图（DAG）。
- **节点（Nodes）**：代表张量（输入、中间变量、输出）。
- **边（Edges）**：代表函数操作（如加法、乘法）。

当你执行 z = y + 5 时，PyTorch 并没有立刻算导数，
- <font color='blue'>它只是在图上画了一条**从 y 到 z **的边，并记录下“这是一个加法操作”。</font>
- 当你调用 backward() 时，系统才会沿着这张图从右向左（**从 z 到 x**）反向遍历，利用链式法则计算梯度。

# 触发求导：backward() 与 grad

- tensor.backward()：这是反向传播的触发器。它从当前张量开始，沿着计算图一路回溯，计算所有 requires_grad=True 的叶子节点的梯度。
- tensor.grad：这是梯度的“收件箱”。<font color='blue'>计算完成后，梯度会被累加并存储在这个属性中。</font>

In [2]:
# 继续上面的代码
z.backward() 

# 根据数学公式 z = x * w + 5
# dz/dx = w = 3.0
# dz/dw = x = 2.0
print(x.grad)  # 输出: tensor([3.])
print(w.grad)  # 输出: tensor([2.])

tensor([3.])
tensor([2.])


# ⚠️ 极其重要的实战避坑指南

在实际训练神经网络时，Autograd 有几个极其容易踩坑的特性：

## 坑一：梯度累积（Gradient Accumulation）

**PyTorch 默认会累加梯度，而不是覆盖。** 
   - <font color='blue'>这意味着如果你连续调用两次 backward()，第二次的梯度会加到第一次的结果上。</font>
- 解决方案：**在每次优化器更新参数前，必须手动清零梯度。**

In [ ]:
optimizer.zero_grad()  # 或者 model.zero_grad()
loss.backward()
optimizer.step()

## 坑二：不需要梯度的计算（推理/验证阶段）
<font color='blue'>在模型评估或测试时，你不需要计算梯度。</font><font color='red'>如果不关闭 Autograd，它会白白消耗大量内存来构建计算图，导致显存溢出（OOM）。</font>
- 解决方案：使用 with torch.no_grad(): 上下文管理器。

In [ ]:
model.eval()
with torch.no_grad():
    predictions = model(test_data)

## 坑三：标量限制
- <font color='red'>backward() 默认只能对标量（只有一个元素的张量，如 Loss）调用。</font>
  - 如果你的输出是一个向量（比如形状为 [32, 10]），你必须提供一个与输出形状相同的 gradient 参数，告诉 PyTorch 如何对向量求导。
- 解决方案：<font color='blue'>通常在深度学习最后，我们都会通过 .sum() 或 .mean() 将 Loss 压缩成标量。</font>

In [ ]:
loss = criterion(outputs, labels)  # 已经是标量
loss.backward() 

## 💡 总结：神经网络训练的 Autograd 闭环
结合我们之前学的张量操作，一个完整的 PyTorch 训练循环本质上是这样的：

1. 前向传播：
   - 数据流过模型，Autograd 默默记录计算图。
2. 计算 Loss：
   - <font color='blue'>得到标量结果。</font>
3. optimizer.zero_grad()：
   - 清空上一步的梯度。
4. loss.backward()：
   - Autograd 沿着计算图反向求导，把梯度存入各个参数的 .grad 中。
5. optimizer.step()：
   - 优化器读取 .grad，更新模型参数。
